# Load USDA dataset with categories

In [1]:
import json
import pandas as pd

In [ ]:
# build a dataframe out of the /FoodData_Central_csv_2025-12-18

In [15]:
import pandas as pd

DATA_DIR = "FoodData_Central_csv_2025-12-18"

# --- load tables --------------------------------------------------------
food = pd.read_csv(f"{DATA_DIR}/food.csv", usecols=["fdc_id", "description", "food_category_id"])
food_nutrient = pd.read_csv(f"{DATA_DIR}/food_nutrient.csv", usecols=["fdc_id", "nutrient_id", "amount"])

# --- nutrient IDs of interest -------------------------------------------
NUTRIENT_MAP = {
    1008: "energy_kcal",
    1003: "protein_g",
    1005: "carbohydrate_g",
    1004: "fat_g",
}

# keep only the four nutrients, then pivot to one column per nutrient
fn = food_nutrient[food_nutrient["nutrient_id"].isin(NUTRIENT_MAP)].copy()
fn["nutrient_name"] = fn["nutrient_id"].map(NUTRIENT_MAP)
fn_pivot = fn.pivot_table(index="fdc_id", columns="nutrient_name", values="amount", aggfunc="first")
fn_pivot.reset_index(inplace=True)

# --- merge with food table & rename category ----------------------------
df = food.merge(fn_pivot, on="fdc_id", how="inner")
df.rename(columns={"food_category_id": "category", "description": "food_description"}, inplace=True)

# --- drop rows where ANY of the five key columns is NaN -----------------
required = ["energy_kcal", "protein_g", "carbohydrate_g", "fat_g", "category"]
df.dropna(subset=required, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(1826573, 7)


,fdc_id,food_description,category,carbohydrate_g,energy_kcal,fat_g,protein_g
0,1105904,WESSON Vegetable Oil 1 GAL,Oils Edible,0.00,867.0,93.33,0.00
1,1105905,SWANSON BROTH BEEF,Herbs/Spices/Extracts,0.42,4.0,0.00,0.83
2,1105906,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,Prepared Soups,6.12,82.0,5.31,2.45
3,1105907,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,Prepared Soups,5.31,82.0,6.12,1.22
4,1105908,SWANSON BROTH CHICKEN,Herbs/Spices/Extracts,0.42,4.0,0.00,0.83


In [2]:
df.category.nunique()

637

In [3]:
df.drop(columns=["fdc_id"], inplace=True)

In [4]:
# list all different unique categories
categories = df.category.unique()
print(categories)

<StringArray>
[                   'Oils Edible',          'Herbs/Spices/Extracts',
                 'Prepared Soups', 'Sauces/Spreads/Dips/Condiments',
   'Dough Based Products / Meals', 'Vegetables  Prepared/Processed',
                          'Bread',               'Biscuits/Cookies',
          'Sweet Bakery Products',        'Savoury Bakery Products',
 ...
                           '7506',                           '7504',
                           '7702',                           '7704',
                           '7804',                           '9204',
                           '7208',                           '9802',
                           '7206',                           '7104']
Length: 637, dtype: str


In [9]:
# print all unique categories with only numbers in the string
categories_numeric = [cat for cat in categories if cat.isdigit()]
print(categories_numeric)
# count them
print(len(categories_numeric))

['18', '23', '19', '11', '24', '25', '14', '4', '7', '1', '3', '5', '10', '8', '16', '15', '9', '20', '12', '13', '22', '21', '2', '6', '17', '1004', '1002', '1006', '1008', '1202', '1902', '1820', '1822', '8412', '5802', '9007', '9010', '1206', '1204', '1208', '1402', '7220', '9404', '9402', '9999', '8008', '8006', '5804', '5502', '1602', '1604', '3602', '2502', '3720', '6432', '2002', '2004', '9008', '2602', '2604', '2006', '3002', '8002', '2008', '2206', '2202', '2204', '2010', '2606', '2608', '2402', '2404', '3404', '3004', '8410', '3006', '8404', '3202', '3402', '3502', '6411', '3740', '3742', '3702', '3704', '3730', '3703', '3804', '3808', '3806', '3706', '2802', '3102', '5004', '3104', '2806', '1904', '8406', '3744', '2804', '3722', '7204', '4202', '4204', '5506', '4206', '3208', '5202', '5504', '4402', '4208', '4404', '5402', '5404', '9012', '5204', '5008', '5002', '5006', '4004', '4804', '4802', '4002', '4602', '4604', '9002', '3506', '3504', '3406', '3204', '3206', '6012', '8

In [ ]:
# --- resolve numeric category IDs to descriptions ----------------------
food_cat = pd.read_csv(f"{DATA_DIR}/food_category.csv", dtype=str)
wweia_cat = pd.read_csv(f"{DATA_DIR}/wweia_food_category.csv", dtype=str)

# build a combined lookup: food_category id + wweia code -> description
cat_map = dict(zip(food_cat["id"], food_cat["description"]))
cat_map.update(dict(zip(wweia_cat["wweia_food_category"], wweia_cat["wweia_food_category_description"])))

In [16]:
# now replace the strings with only numbers in the "category" column in df with the resolved descriptions
df["category"] = df["category"].apply(lambda x: cat_map.get(x, x))

In [17]:
df.category

0                                              Oils Edible
1                                    Herbs/Spices/Extracts
2                                           Prepared Soups
3                                           Prepared Soups
4                                    Herbs/Spices/Extracts
                                ...                       
1826568                     Sauces/Spreads/Dips/Condiments
1826569                     Sauces/Spreads/Dips/Condiments
1826570                     Sauces/Spreads/Dips/Condiments
1826571                   Vegetable Based Products / Meals
1826572    Meat/Poultry/Other Animals - Prepared/Processed
Name: category, Length: 1826573, dtype: str

In [22]:
usda_data = df.copy()
usda_data.to_csv("usda_data_cats.csv", index=False)

# Load open food facts with categories

In [1]:
import pandas as pd

cols = [
    "product_name",
    "main_category_en",
    "categories",
    "categories_en",
    "energy-kcal_100g",
    "fat_100g",
    "carbohydrates_100g",
    "proteins_100g",
]

off = pd.read_csv(
    "en.openfoodfacts.org.products.csv",
    sep="\t",
    usecols=cols,
    low_memory=False,
)

off.dropna(subset=cols, inplace=True)
off.reset_index(drop=True, inplace=True)

print(off.shape)
off.head()

(43326, 8)


,product_name,categories,categories_en,main_category_en,energy-kcal_100g,fat_100g,carbohydrates_100g,proteins_100g
0,greek yogurt,"Dairies, Fermented foods, Desserts, Fermented ...","Dairies,Fermented foods,Fermented milk product...",Greek-style yogurts,86.666667,1.6666666666667,10.666667,7.333333
1,Shake Mix Vanilla Flavour,Dietary supplements,Dietary supplements,Dietary supplements,376.000000,7.2,42.000000,36.000000
2,German fine bread,"Plant-based foods and beverages, Plant-based f...","Plant-based foods and beverages,Plant-based fo...",Breads,263.000000,3.51,52.630000,7.020000
3,Harvest whole wheat bread,"Plant-based foods and beverages, Plant-based f...","Plant-based foods and beverages,Plant-based fo...",Chouquettes,232.558140,4.6511627906977,41.860465,11.627907
4,Sourdough Bread,"Plant-based foods and beverages, Plant-based f...","Plant-based foods and beverages,Plant-based fo...",Sliced breads,214.285714,1.403875,59.118050,8.390992


In [4]:
off.main_category_en.nunique()

6368

In [5]:
off.to_csv("off_data_cats.csv", index=False)

In [8]:
usda_data = pd.read_csv("usda_data_cats.csv")

In [9]:
usda_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1826573 entries, 0 to 1826572
Data columns (total 7 columns):
 #   Column            Dtype  
---  ------            -----  
 0   fdc_id            int64  
 1   food_description  str    
 2   category          str    
 3   carbohydrate_g    float64
 4   energy_kcal       float64
 5   fat_g             float64
 6   protein_g         float64
dtypes: float64(4), int64(1), str(2)
memory usage: 97.5 MB


In [10]:
usda_data.columns

Index(['fdc_id', 'food_description', 'category', 'carbohydrate_g',
       'energy_kcal', 'fat_g', 'protein_g'],
      dtype='str')

In [17]:
# make the columns match ['item_id', 'item_name', 'brand', 'kcal_100g', 'protein_100g',
#       'carbs_100g', 'fat_100g']
# and keep the added columns (which do not match the list above)
usda_data.rename(
    columns={
        "food_description": "item_name",
        "category": "cat",
        "energy_kcal": "kcal_100g",
        "protein_g": "protein_100g",
        "carbohydrate_g": "carbs_100g",
        "fat_g": "fat_100g",
    },
    inplace=True,
)

# also for off data
off.rename(
    columns={
        "product_name": "item_name",
        "main_category_en": "cat",
        "energy-kcal_100g": "kcal_100g",
        "proteins_100g": "protein_100g",
        "carbohydrates_100g": "carbs_100g",
        "fat_100g": "fat_100g",
    },
    inplace=True,
)


In [19]:
# add a column to each dataframe called "source" with values "off" and "usda" respectively
off["source"] = "off"
usda_data["source"] = "usda"

In [20]:
off_clean = off.copy()
usda_clean = usda_data.copy()
off_clean.drop(columns=["categories", "categories_en"], inplace=True)
usda_clean.drop(columns=["fdc_id"], inplace=True)

In [22]:
print(off_clean.columns)
print(usda_clean.columns)

Index(['item_name', 'cat', 'kcal_100g', 'fat_100g', 'carbs_100g',
       'protein_100g', 'source'],
      dtype='str')
Index(['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g',
       'protein_100g', 'source'],
      dtype='str')


In [23]:
off_clean.to_csv("off_data_clean2.csv", index=False)
usda_clean.to_csv("usda_data_clean2.csv", index=False)

In [33]:
# show all off_clean.cat where "fr:" is anywhere in the string
off_clean.cat[off_clean.cat.str.contains('fr:')].nunique()

641

In [32]:
# Find all language-tag prefixes in the cat column
import pandas as pd

# Extract 2-letter prefix before ":"
prefixes = off_clean["cat"].str.extract(r"^([a-z]{2}):", expand=False).dropna()

print(f"Rows with language tag: {len(prefixes)} / {len(off_clean)}")
print(f"\nUnique prefixes ({prefixes.nunique()}):")
print(prefixes.value_counts().to_string())

Rows with language tag: 2330 / 43326

Unique prefixes (39):
cat
fr    958
de    435
es    239
nl    137
ro    124
it    120
pt     47
pl     40
ru     28
el     24
uk     20
tr     19
cs     15
sv     15
da     13
bg     12
nb     11
hr     11
ja      8
th      8
fi      6
hu      5
ar      4
sk      4
lt      4
ca      3
lv      3
sl      2
zh      2
sr      2
vi      2
id      2
et      1
cy      1
ka      1
no      1
he      1
mt      1
ko      1


In [34]:
# drop all the rows where "cat" contains a language tag (e.g. "fr:") since we cannot resolve those categories
off_clean = off_clean[~off_clean["cat"].str.contains(r"^[a-z]{2}:")].copy()
off_clean.reset_index(drop=True, inplace=True)

In [37]:
off_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 40996 entries, 0 to 40995
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_name     40996 non-null  str    
 1   cat           40996 non-null  str    
 2   kcal_100g     40996 non-null  float64
 3   fat_100g      40996 non-null  str    
 4   carbs_100g    40996 non-null  float64
 5   protein_100g  40996 non-null  float64
 6   source        40996 non-null  str    
dtypes: float64(3), str(4)
memory usage: 2.2 MB


In [38]:
# make all the entries lowercase in the 'cat' and 'item_name' columns for both dataframes
off_clean["cat"] = off_clean["cat"].str.lower()
off_clean["item_name"] = off_clean["item_name"].str.lower()
usda_clean["cat"] = usda_clean["cat"].str.lower()
usda_clean["item_name"] = usda_clean["item_name"].str.lower()

In [46]:
# count the number of rows in off_clean where "cat" contains a space (indicating multiple words)
off_word_counts = off_clean.copy()
off_word_counts['word_count'] = off_word_counts['cat'].str.count(' ') + 1

print(off_word_counts['word_count'].value_counts().to_string())

word_count
1     22399
2     13263
3      3322
4      1123
5       487
6       165
7        98
8        76
10       20
11       17
9        16
13        6
12        2
14        1
18        1


In [44]:
# count the number of rows in off_clean where "cat" contains a space (indicating multiple words)
usda_word_counts = usda_clean.copy()
usda_word_counts['word_count'] = usda_word_counts['cat'].str.count(' ') + 1

print(usda_word_counts['word_count'].value_counts().to_string())

word_count
1      424265
4      330371
5      303528
3      271277
2      262115
6      173025
8       37872
7       23688
11        118
29         65
14         53
13         40
75         25
40         19
9          17
98         14
109        12
32         11
74         10
42          6
93          6
50          5
94          5
63          5
39          4
89          4
43          3
12          3
90          2
30          1
72          1
67          1
83          1
76          1


In [50]:
off_word_counts[off_word_counts['word_count'] == 7].nunique()


item_name       96
cat             35
kcal_100g       66
fat_100g        48
carbs_100g      53
protein_100g    48
source           1
word_count       1
dtype: int64

- Most of the higher word-count categories have only a few matching items
- category has more information about the item, then the item name

In [61]:
usda_word_counts[usda_word_counts['word_count'] == 6].nunique()
#now print the 11 unique categories in usda_clean where the word count is 7
print(usda_clean[usda_word_counts['word_count'] == 6]['cat'].unique())


<StringArray>
[                                            'frozen fruit & fruit juice concentrates',
                                            'popcorn, peanuts, seeds & related snacks',
                                     'seasoning mixes, salts, marinades & tenderizers',
                                                'ketchup, mustard, bbq & cheese sauce',
                                       'frozen breakfast sandwiches, biscuits & meals',
                                                'french fries, potatoes & onion rings',
                                                     'pizza mixes & other dry dinners',
                                            'energy, protein & muscle recovery drinks',
                                            'pancakes, waffles, french toast & crepes',
                          'chips/crisps/snack mixes - natural/extruded (shelf stable)',
                                                   'drinks flavoured - ready to drink',
                  

In [62]:
usda_word_counts.item_name[usda_word_counts['word_count'] == 6].head(20)


172    mixed fruit medley freshly frozen strawberries...
173                               walnut halves & pieces
203                             garden vegetable dip mix
215           georgia mustard bbq sauce, georgia mustard
233                                          blueberries
247       burritos, egg, applewood smoked bacon & cheese
251    organic morning fruit & nut granola mix, purpl...
270                             taco seasoning mix, taco
271                                    green chile sauce
276                                          raw almonds
286    blueberry flavored oven roasted almonds, blueb...
296                                    sunflower kernels
318                                             fish fry
330    sweet potato fries french fried potatoes, swee...
331               shoestring fries french fried potatoes
338                              seasoned mushroom curry
389    olde cape cod, bbq & grilling sauce, honey, or...
390    olde cape cod, bbq & gri

In [57]:
usda_clean.cat[usda_clean.cat.str.contains('cereal')].unique()

<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       'cereal',
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

There are f.e. 20 different cereal categories in the usda dataset. 

In [58]:
off_clean.cat[off_clean.cat.str.contains('cereal')].unique()
# 84 in the off-dataset

<StringArray>
[                                                                                                                                                                       'yogurts with cereals',
                                                                                                                                                                           'breakfast cereals',
                                                                                                                                                                               'cereal flakes',
                                                                                                                                                                                 'cereal bars',
                                                                                                                                                                            'extruded cereals',
                          